# WORK9 — 06A Prospective Frozen Forecast V01

This notebook creates the **immutable June-2026 forecast vintage** for the reserved unseen holdout:

- H1 → July 2026
- H2 → August 2026
- H3 → September 2026

Champion V1 is already frozen:

$$\hat y = P(Y>0\mid X) \times E[Y\mid Y>0,X]$$

This notebook **does not query Supabase and does not read future actuals**. It trains only from the accepted Dataset V012 Pair panel whose maximum month must be exactly `2026-06-01`.

Use a GPU runtime if available, then **Run all**.


In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install lightgbm pyarrow pyyaml pytest

from pathlib import Path
import json, sys, subprocess, datetime
import pandas as pd
import yaml

ROOT = Path('/content/drive/MyDrive/work9')
SRC = ROOT / '02_src/modeling/prospective_frozen_forecast_runner_v01.py'
TEST = ROOT / '07_tests/test_prospective_frozen_forecast_runner_v01.py'
FF_CONTRACT = ROOT / '01_config/prospective_frozen_forecast_contract_v01.yaml'
MODEL_CONTRACT = ROOT / '01_config/model_contract_v02.yaml'
ARCH_CONTRACT = ROOT / '01_config/model_architecture_freeze_v01.yaml'
ARCH_DOC = ROOT / '00_docs/MODEL_ARCHITECTURE_FREEZE_V1.0.md'
RESERVATION_DOC = ROOT / '00_docs/FROZEN_TEST_RESERVATION_V1.0.md'
DATA_PTR = ROOT / '01_config/current_dataset_run.json'
SEL_PTR = ROOT / '01_config/current_feature_selection_run.json'
ARCH_PTR = ROOT / '01_config/current_model_architecture_freeze.json'

for p in [SRC, TEST, FF_CONTRACT, MODEL_CONTRACT, ARCH_CONTRACT, ARCH_DOC, RESERVATION_DOC, DATA_PTR, SEL_PTR, ARCH_PTR]:
    assert p.exists(), f'Missing: {p}'
print('06A inputs found')


Mounted at /content/drive
06A inputs found


## 1. Verify the frozen architecture and untouched source cutoff

The accepted Pair panel must stop at June 2026. If a later month is present, this notebook aborts rather than silently contaminating the prospective vintage.


In [2]:
dataset = json.loads(DATA_PTR.read_text(encoding='utf-8'))
selection = json.loads(SEL_PTR.read_text(encoding='utf-8'))
freeze = json.loads(ARCH_PTR.read_text(encoding='utf-8'))
ff_contract = yaml.safe_load(FF_CONTRACT.read_text(encoding='utf-8'))

assert dataset['status'] == 'PASS' and dataset['dataset_version'] == 'dataset_v012'
assert selection['status'] == 'PASS' and selection['selection_version'] == 'feature_selection_v04'
assert freeze['status'] == 'APPROVED'
assert freeze['freeze_scope'] == 'ARCHITECTURE_ONLY'
assert freeze['champion_model'] == 'soft_two_part_expected'
assert freeze['frozen_test_open'] is False
assert ff_contract['forecast_origin'] == '2026-06-01'
assert ff_contract['target_months'] == {'H1':'2026-07-01','H2':'2026-08-01','H3':'2026-09-01'}

PAIR_PANEL = Path(dataset['pair_panel_path'])
SELECTED = Path(selection['pair_selected_path'])
for p in [PAIR_PANEL, SELECTED]:
    assert p.exists(), p

months = pd.read_parquet(PAIR_PANEL, columns=['month'])['month']
max_month = pd.to_datetime(months).dt.to_period('M').dt.to_timestamp().max()
assert max_month == pd.Timestamp('2026-06-01'), f'Pair panel max month is {max_month}; prospective freeze requires 2026-06-01'

print('Architecture freeze:', freeze['freeze_id'])
print('Champion:', freeze['champion_model'])
print('Dataset:', dataset['run_id'])
print('Feature Selection:', selection['run_id'])
print('Pair panel max month:', max_month.date())
print('Reserved targets: 2026-07 / 2026-08 / 2026-09')
print('NO Supabase query is executed by this notebook.')


Architecture freeze: model_architecture_freeze_v01_20260815T152950Z
Champion: soft_two_part_expected
Dataset: core_dataset_v012_20260815T122509Z
Feature Selection: feature_selection_v04_20260815T130048Z
Pair panel max month: 2026-06-01
Reserved targets: 2026-07 / 2026-08 / 2026-09
NO Supabase query is executed by this notebook.


## 2. Unit tests

Tests cover the soft expected-demand formula, strict cutoff guard, calibration boundaries, and same-universe summary behavior.


In [3]:
r = subprocess.run([sys.executable, '-m', 'pytest', '-q', str(TEST)], text=True, capture_output=True)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr)
    raise AssertionError(f'06A unit tests failed (returncode={r.returncode})')
print('06A unit tests PASS')


........                                                                 [100%]
8 passed in 2.69s

06A unit tests PASS


## 3. Fit the frozen V1 architecture at origin June 2026

For each H1/H2/H3:

- FIT uses target-available history before Apr 2026;
- CAL uses Apr–Jun 2026 current-active known-Pair rows only for iteration selection;
- the final component models are refit under the already-frozen architecture;
- forecasts are created only for current-active known Pairs at the June origin.


In [4]:
import importlib.util
spec = importlib.util.spec_from_file_location('ff', SRC)
ff = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ff)

run_id = 'prospective_frozen_forecast_v01_' + datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
run_dir = ROOT / '08_runs' / run_id
report_dir = ROOT / '06_reports/prospective_frozen_forecast' / run_id

manifest = ff.run_prospective_frozen_forecast(
    pair_panel_path=str(PAIR_PANEL),
    selected_feature_path=str(SELECTED),
    model_contract_path=str(MODEL_CONTRACT),
    frozen_forecast_contract_path=str(FF_CONTRACT),
    architecture_freeze_contract_path=str(ARCH_CONTRACT),
    architecture_freeze_doc_path=str(ARCH_DOC),
    dataset_pointer=dataset,
    selection_pointer=selection,
    architecture_freeze_pointer=freeze,
    run_dir=str(run_dir),
    report_dir=str(report_dir),
    run_id=run_id,
    work9_root=str(ROOT),
)

print('06A PASS:', run_id)
print('Forecast origin:', manifest['forecast_origin'])
print('Target months:', manifest['target_months'])
print('Unique forecast Pairs:', manifest['unique_pairs'])
print('Forecast rows:', manifest['forecast_rows'])


/content/drive/MyDrive/work9/02_src/modeling/prospective_frozen_forecast_runner_v01.py:126: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  x["target_available"] = x["target_available"].fillna(False).astype(bool)
/content/drive/MyDrive/work9/02_src/modeling/prospective_frozen_forecast_runner_v01.py:126: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  x["target_available"] = x["target_available"].fillna(False).astype(bool)
/content/drive/MyDrive/work9/02_src/modeling/prospective_frozen_forecast_runner_v01.py:126: FutureWarning: Downcasting object dtype arrays

06A PASS: prospective_frozen_forecast_v01_20260815T155226Z
Forecast origin: 2026-06-01
Target months: ['2026-07-01', '2026-08-01', '2026-09-01']
Unique forecast Pairs: 17253
Forecast rows: 51759


## 4. Inspect the locked forecast vintage

These are forecasts only. There are no Jul/Aug/Sep actual columns in this artifact.


In [5]:
pred_path = Path(manifest['artifacts']['prediction_parquet'])
summary_path = Path(manifest['artifacts']['summary_csv'])
audit_path = Path(manifest['artifacts']['training_audit_csv'])

pred = pd.read_parquet(pred_path)
summary = pd.read_csv(summary_path)
audit = pd.read_csv(audit_path)

assert not any(c.startswith('target_actual') for c in pred.columns)
assert pred['forecast_origin'].nunique() == 1
assert pd.Timestamp(pred['forecast_origin'].iloc[0]) == pd.Timestamp('2026-06-01')
assert sorted(pd.to_datetime(pred['target_month']).dt.strftime('%Y-%m-%d').unique().tolist()) == ['2026-07-01','2026-08-01','2026-09-01']

print('Forecast summary:')
display(summary)
print('Training / calibration audit:')
display(audit)
print('Prediction sample:')
display(pred.head(10))


Forecast summary:


,horizon,target_month,n_pairs,n_base_sku,n_branch,forecast_sum_m2,forecast_mean_m2,mean_p_positive,mean_positive_quantity_m2
0,1,2026-07-01,17253,1447,58,469830.323620,27.231805,0.299377,65.867457
1,2,2026-08-01,17253,1447,58,396152.269628,22.961356,0.262922,62.671243
2,3,2026-09-01,17253,1447,58,318297.385404,18.448814,0.236269,57.685762


Training / calibration audit:


,horizon,calibration_start_target_month,fit_rows,calibration_rows,fit_target_max,calibration_target_min,calibration_target_max,forecast_rows,positive_fit_rows,positive_calibration_rows,occurrence_n_estimators,positive_quantity_n_estimators,occurrence_device,positive_quantity_device
0,1,2026-04-01,466913,48370,2026-03-01,2026-04-01,2026-06-01,17253,130647,15614,139,184,cpu,cpu
1,2,2026-04-01,441168,46897,2026-03-01,2026-04-01,2026-06-01,17253,117909,14696,108,148,cpu,cpu
2,3,2026-04-01,415246,45929,2026-03-01,2026-04-01,2026-06-01,17253,105435,14138,95,143,cpu,cpu


Prediction sample:


,base_sku,branch_code,forecast_origin,target_month,horizon,known_pair_asof_origin,current_production_forecast_mask,p_positive,pred_positive_quantity,forecast_m2
0,00.L1.60120.G12P18,000,2026-06-01,2026-07-01,1,True,True,0.056765,168.203569,9.548014
1,00.L1.60120.G12P18,000,2026-06-01,2026-08-01,2,True,True,0.051752,164.739485,8.525667
2,00.L1.60120.G12P18,000,2026-06-01,2026-09-01,3,True,True,0.048170,118.053092,5.686606
3,00.L1.8080.P87625N,015,2026-06-01,2026-07-01,1,True,True,0.011078,52.806410,0.584999
4,00.L1.8080.P87625N,015,2026-06-01,2026-08-01,2,True,True,0.016047,53.086277,0.851894
5,00.L1.8080.P87625N,015,2026-06-01,2026-09-01,3,True,True,0.018887,51.777990,0.977949
6,02.L1.6060.7250,000,2026-06-01,2026-07-01,1,True,True,0.009221,72.474724,0.668258
7,02.L1.6060.7250,000,2026-06-01,2026-08-01,2,True,True,0.015464,49.819967,0.770407
8,02.L1.6060.7250,000,2026-06-01,2026-09-01,3,True,True,0.018887,59.697680,1.127531
9,02.L1.6060.7341,000,2026-06-01,2026-07-01,1,True,True,0.009221,72.172669,0.665472


## 5. Safety gate

The forecast vintage can be locked only if no future labels were read, no threshold/bias scaling/rounding was introduced, and evaluation has not run.


In [6]:
s = manifest['safety']
assert s['supabase_accessed'] is False
assert s['future_actual_labels_read'] is False
assert s['future_actual_labels_used_for_fit_or_calibration'] is False
assert s['pair_panel_max_month_equals_forecast_origin'] is True
assert s['production_universe_current_active_known_pair_only'] is True
assert s['hard_zero_threshold_used'] is False
assert s['posthoc_global_bias_scaling_used'] is False
assert s['pair_level_rounding_used'] is False
assert s['architecture_changed_after_freeze'] is False
assert s['frozen_test_evaluation_run'] is False
assert s['production_published'] is False

print(json.dumps(s, indent=2))
print('Safety gate PASS')


{
  "supabase_accessed": false,
  "future_actual_labels_read": false,
  "future_actual_labels_used_for_fit_or_calibration": false,
  "pair_panel_max_month_equals_forecast_origin": true,
  "production_universe_current_active_known_pair_only": true,
  "hard_zero_threshold_used": false,
  "posthoc_global_bias_scaling_used": false,
  "pair_level_rounding_used": false,
  "architecture_changed_after_freeze": false,
  "frozen_test_evaluation_run": false,
  "production_published": false
}
Safety gate PASS


## 6. Publish the frozen forecast pointer

This pointer means **the prediction vintage is locked**. It does not mean the Frozen Test has been scored. Formal evaluation waits until Jul–Aug–Sep 2026 are all closed/loaded.


In [7]:
assert manifest['status'] == 'PASS'
reservation = json.loads(Path(manifest['artifacts']['reservation_json']).read_text(encoding='utf-8'))
assert reservation['status'] == 'LOCKED_PENDING_ACTUALS'

ptr = {
    'run_id': run_id,
    'status': 'PASS',
    'frozen_forecast_version': 'prospective_frozen_forecast_v01',
    'architecture_freeze_id': freeze['freeze_id'],
    'champion_model': 'soft_two_part_expected',
    'forecast_origin': '2026-06-01',
    'target_months': ['2026-07-01','2026-08-01','2026-09-01'],
    'evaluation_status': 'LOCKED_PENDING_ACTUALS',
    'run_dir': str(run_dir),
    'report_dir': str(report_dir),
    'prediction_path': manifest['artifacts']['prediction_parquet'],
    'prediction_sha256': manifest['output_sha256']['prediction_parquet'],
    'manifest_path': str(run_dir / 'prospective_frozen_forecast_manifest.json'),
    'reservation_path': manifest['artifacts']['reservation_json'],
}
ptr_path = ROOT / '01_config/current_prospective_frozen_forecast_run.json'
ptr_path.write_text(json.dumps(ptr, ensure_ascii=False, indent=2), encoding='utf-8')

print('06A PROSPECTIVE FROZEN FORECAST LOCKED')
print('Pointer:', ptr_path)
print('Prediction SHA256:', ptr['prediction_sha256'])
print('STOP HERE. Do not score Jul/Aug/Sep until all three months are closed/loaded.')


06A PROSPECTIVE FROZEN FORECAST LOCKED
Pointer: /content/drive/MyDrive/work9/01_config/current_prospective_frozen_forecast_run.json
Prediction SHA256: 3dd59e503f3bf59eaa542222a20995aca9af61732eaecb4702e61cb1161f33dc
STOP HERE. Do not score Jul/Aug/Sep until all three months are closed/loaded.
